In [1]:
from google import genai


# ============================================================
# 1. GEMINI API KEY
# ============================================================

GEMINI_API_KEY = "apikey"

client = genai.Client(
    api_key=GEMINI_API_KEY
)


# ============================================================
# 2. AI TEACHER INSTRUCTIONS
# ============================================================

TEACHER_INSTRUCTIONS = """
You are an expert AI Teacher.

Your job is to teach students, not just provide answers.

Rules:
- Explain concepts using simple layman terms.
- Student level is Beginner.
- Answer exactly 3 short lines.
- Be precise.
- Avoid unnecessary technical jargon.
- Explain technical terms simply when needed.
"""


# ============================================================
# 3. CONVERSATION MEMORY
# ============================================================

conversation_history = []


# ============================================================
# 4. AI TEACHER FUNCTION
# ============================================================

def ask_teacher(
    question,
    subject="AI & Data Science",
    student_level="Beginner",
    language="English"
):

    user_message = f"""
Subject: {subject}
Student Level: {student_level}
Preferred Language: {language}

Student Question:
{question}
"""

    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    # Build conversation
    conversation_text = ""

    for message in conversation_history:

        conversation_text += (
            message["role"].upper()
            + ":\n"
            + message["content"]
            + "\n\n"
        )

    # ========================================================
    # GEMINI API
    # ========================================================

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=conversation_text,
        config={
            "system_instruction": TEACHER_INSTRUCTIONS,
            "temperature": 0.2,
            "max_output_tokens": 500
        }
    )

    answer = response.text.strip()

    # Save response
    conversation_history.append({
        "role": "assistant",
        "content": answer
    })

    return answer


# ============================================================
# 5. CHAT APPLICATION
# ============================================================

def start_teacher():

    print("=" * 60)
    print("                    AI TEACHER")
    print("=" * 60)

    print("Subject : AI & Data Science")
    print("Level   : Beginner")
    print("Language: English")

    print("\nType 'exit' to stop.")
    print("Type 'clear' to clear conversation memory.")

    print("=" * 60)

    while True:

        question = input("\nStudent: ").strip()

        # Exit
        if question.lower() == "exit":

            print("\nTeacher: Keep learning! 👋")
            break

        # Clear memory
        if question.lower() == "clear":

            conversation_history.clear()

            print("\nTeacher: Conversation memory cleared.")
            continue

        # Empty question
        if not question:

            print("\nTeacher: Please ask a question.")
            continue

        # Ask teacher
        try:

            answer = ask_teacher(
                question=question,
                subject="AI & Data Science",
                student_level="Beginner",
                language="English"
            )

            print("\nTeacher:")
            print(answer)

            print("\n" + "-" * 60)

        except Exception as e:

            print("\nError:")
            print(e)


# ============================================================
# 6. RUN
# ============================================================

if __name__ == "__main__":
    start_teacher()

                    AI TEACHER
Subject : AI & Data Science
Level   : Beginner
Language: English

Type 'exit' to stop.
Type 'clear' to clear conversation memory.



Student:  what is ML


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



Teacher:
Machine Learning (ML) is teaching computers to learn from experience, just like humans do.
Instead of writing rules for everything, you feed the computer data so it can discover patterns on its own.
Once it learns these patterns, it can make smart decisions or predictions on brand-new information.

------------------------------------------------------------



Student:  exit



Teacher: Keep learning! 👋


In [3]:
#!pip install -U google-genai

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   -------------------------- ------------- 0.8/1.2 MB 4.8 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 4.5 MB/s  0:00:00
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.22.0
    Uninstalling google-genai-2.22.0:
      Successfully uninstalled google-genai-2.22.0


# Stage2 -  input Quries- Generate

In [4]:
###### import pandas as pd
from google import genai


# ============================================================
# 1. GEMINI API KEY
# ============================================================

GEMINI_API_KEY = "Apikey"

client = genai.Client(
    api_key=GEMINI_API_KEY
)


# ============================================================
# 2. FILE CONFIGURATION
# ============================================================

INPUT_CSV = "ai_teacher_20_questions.csv"
OUTPUT_CSV = "ai_teacher_results.csv"


# ============================================================
# 3. LOAD DATASET
# ============================================================

df = pd.read_csv(INPUT_CSV)

print("Dataset loaded")
print(f"Number of questions: {len(df)}")


# ============================================================
# 4. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "User Query",
    "Teacher Response"
]

for column in required_columns:

    if column not in df.columns:
        raise ValueError(
            f"Missing required column: {column}"
        )


# ============================================================
# 5. AI TEACHER INSTRUCTIONS
# ============================================================

TEACHER_INSTRUCTIONS = """
You are an expert AI Teacher.

Your job is to teach students, not just provide answers.

Teaching rules:
- Explain the concept using simple layman terms.
- Assume the student is a beginner.
- Answer should be precise.
- Give exactly 3 short lines.
- Avoid unnecessary technical jargon.
- Explain technical terms simply when necessary.

Response format:
Explain the concept simply.
"""


# ============================================================
# 6. GENERATE ANSWER
# ============================================================

def generate_answer(question):

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=question,
        config={
            "system_instruction": TEACHER_INSTRUCTIONS,
            "temperature": 0.2,
            "max_output_tokens": 500
        }
    )

    return response.text.strip()


# ============================================================
# 7. GENERATE ANSWERS FOR ALL QUESTIONS
# ============================================================

generated_answers = []

for index, row in df.iterrows():

    question = row["User Query"]

    print("\n" + "=" * 70)
    print(f"Processing {index + 1}/{len(df)}")
    print(f"Question: {question}")

    try:

        answer = generate_answer(question)

        print("Generated Answer:")
        print(answer)

    except Exception as e:

        answer = f"ERROR: {str(e)}"

        print(answer)

    generated_answers.append(answer)


# ============================================================
# 8. ADD GENERATED ANSWERS
# ============================================================

df["GeneratedAnswer"] = generated_answers


# ============================================================
# 9. SELECT FINAL COLUMNS
# ============================================================

df = df[
    [
        "User Query",
        "Teacher Response",
        "GeneratedAnswer"
    ]
]


# ============================================================
# 10. SAVE RESULT
# ============================================================

df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 11. DISPLAY RESULT
# ============================================================

print("\n" + "=" * 70)
print("AI TEACHER TEST COMPLETED")
print("=" * 70)
print(df.to_string(index=False))

print("\nResult saved to:")
print(OUTPUT_CSV)

Dataset loaded
Number of questions: 20

Processing 1/20
Question: What is AI?
ERROR: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

Processing 2/20
Question: How does AI differ from traditional programming?
Generated Answer:
In traditional programming, humans write strict step-by-step rules for the computer to follow

Processing 3/20
Question: What is Machine Learning?
Generated Answer:
Machine learning is teaching computers to learn from experience instead of following strict, pre-written rules.  
Just like a child learns to recognize a dog by seeing many dogs, a computer learns by finding patterns in examples.  
Once it understands these patterns, it can make smart decisions or predictions on new information all by itself.

Processing 4/20
Question: What is Deep Learning?
ERROR: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This m

In [3]:
import pandas as pd

In [ ]:
!pip install -U google-genai pandas

# Stage3 - LLM Evaluation- Answer Quality

In [8]:
# ============================================================
# STAGE 3 - LLM ANSWER QUALITY EVALUATION USING DEEPEVAL
# ============================================================

import pandas as pd

from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric, GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.evaluate import AsyncConfig


# ============================================================
# 1. LOAD DATA
# ============================================================

INPUT_FILE = "ai_teacher_results.csv"

df = pd.read_csv(INPUT_FILE)

print("=" * 70)
print("DATASET LOADED")
print("=" * 70)

print(f"Number of rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")


# ============================================================
# 2. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "User Query",
    "Teacher Response",
    "GeneratedAnswer"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("\nAll required columns found.")


# ============================================================
# 3. REMOVE EMPTY ROWS
# ============================================================

df = df.dropna(
    subset=[
        "User Query",
        "Teacher Response",
        "GeneratedAnswer"
    ]
).reset_index(drop=True)

print(f"Valid questions for evaluation: {len(df)}")


# ============================================================
# 4. CREATE DEEPEVAL TEST CASES
# ============================================================

test_cases = []

for _, row in df.iterrows():

    test_case = LLMTestCase(
        input=str(row["User Query"]),
        actual_output=str(row["GeneratedAnswer"]),
        expected_output=str(row["Teacher Response"])
    )

    test_cases.append(test_case)


print(f"Test cases created: {len(test_cases)}")


# ============================================================
# 5. ANSWER CORRECTNESS METRIC
# ============================================================

correctness = GEval(
    name="Answer Correctness",

    model="gpt-4.1",

    criteria="""
    Evaluate whether the generated answer is factually correct
    compared with the expected teacher answer.

    Give a high score when the important meaning and facts are correct.

    Penalize:
    - incorrect facts
    - contradictory information
    - missing important information

    Minor wording differences should not reduce the score.

    Focus on factual correctness and meaning rather than
    exact wording.
    """,

    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT
    ],

    # IMPORTANT:
    # async_mode must be True or False
    async_mode=False
)


# ============================================================
# 6. ANSWER RELEVANCY METRIC
# ============================================================

relevancy = AnswerRelevancyMetric(
    model="gpt-4.1",

    include_reason=True,

    # IMPORTANT:
    # async_mode must be True or False
    async_mode=False
)


# ============================================================
# 7. HELPFULNESS METRIC
# ============================================================

helpfulness = GEval(
    name="Helpfulness",

    model="gpt-4.1",

    criteria="""
    Evaluate how helpful the generated answer is for a
    beginner student.

    A helpful answer should:

    1. Directly answer the student's question.
    2. Be easy to understand.
    3. Provide enough explanation.
    4. Avoid unnecessary information.
    5. Help the student understand the concept.

    Give a high score when the answer is:

    - clear
    - beginner-friendly
    - relevant
    - educational
    - concise
    """,

    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT
    ],

    # IMPORTANT:
    # async_mode must be True or False
    async_mode=False
)


# ============================================================
# 8. RUN DEEPEVAL
# ============================================================

print("\n" + "=" * 70)
print("STARTING DEEPEVAL EVALUATION")
print("=" * 70)

print("\nMetrics being evaluated:")
print("1. Answer Correctness")
print("2. Answer Relevancy")
print("3. Helpfulness")

print("\nPlease wait...\n")


result = evaluate(
    test_cases=test_cases,

    metrics=[
        correctness,
        relevancy,
        helpfulness
    ],

    # IMPORTANT FOR JUPYTER / ANACONDA
    # Run evaluation synchronously
    async_config=AsyncConfig(
        run_async=False
    )
)


# ============================================================
# 9. DISPLAY FINAL RESULT
# ============================================================

print("\n" + "=" * 70)
print("DEEPEVAL EVALUATION COMPLETED")
print("=" * 70)

print(result)

DATASET LOADED
Number of rows: 20
Columns: ['User Query', 'Teacher Response', 'GeneratedAnswer']

All required columns found.
Valid questions for evaluation: 20
Test cases created: 20

STARTING DEEPEVAL EVALUATION

Metrics being evaluated:
1. Answer Correctness
2. Answer Relevancy
3. Helpfulness

Please wait...



✨ You're running DeepEval's latest Answer Correctness [GEval] Metric! (using gpt-4.1, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1, strict=False, async_mode=False)...

✨ You're running DeepEval's latest Helpfulness [GEval] Metric! (using gpt-4.1, strict=False, async_mode=False)...

Output()

RetryError: RetryError[<Future at 0x1765e600070 state=finished raised RateLimitError>]

In [4]:
%pip install -U deepeval

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   --------------------- ------------------ 0.8/1.5 MB 4.8 MB/s eta 0:00:01
   ---------------------------- ----------- 1.0/1.5 MB 5.0 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 2.5 MB/s  0:00:00
   ---------------------------------------- 0.0/818.8 kB ? eta -:--:--
   ------------------------- -------------- 524.3/818.8 kB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 818.8/818.8 kB 3.3 MB/s  0:00:00
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.8 MB 4.2 MB/s eta 0:00:01
   ----------------------------- ---------- 1.3/1.8 MB 3.7 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 3.3 MB/s  0:00:00

   -- -------------------------------------  1/17 [pyfiglet]
   -- -------------------------------------  1/17 [pyfiglet]
   -- -------------------------------------  1/17 [pyfiglet]


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.21.0 requires click>=8.4.0, but you have click 8.3.3 which is incompatible.


In [6]:
dir(result)

NameError: name 'result' is not defined

In [7]:
import deepeval

print("DeepEval installed successfully")
print("Version:", deepeval.__version__)

DeepEval installed successfully
Version: 4.2.8


In [8]:
import os

if os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY is configured")
else:
    print("OPENAI_API_KEY is NOT configured")

OPENAI_API_KEY is configured


In [9]:
import pandas as pd

from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric, GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.evaluate import AsyncConfig


# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv("ai_teacher_results.csv")

print("=" * 70)
print("DATASET LOADED")
print("=" * 70)

print("Number of questions:", len(df))
print("Columns:", df.columns.tolist())


# ============================================================
# 2. CHECK COLUMNS
# ============================================================

required_columns = [
    "User Query",
    "Teacher Response",
    "GeneratedAnswer"
]

for column in required_columns:
    if column not in df.columns:
        raise ValueError(
            f"Missing required column: {column}"
        )

print("\nAll required columns found.")


# ============================================================
# 3. CREATE TEST CASES
# ============================================================

test_cases = []

for _, row in df.iterrows():

    test_case = LLMTestCase(
        input=str(row["User Query"]),
        actual_output=str(row["GeneratedAnswer"]),
        expected_output=str(row["Teacher Response"])
    )

    test_cases.append(test_case)

print("Test cases created:", len(test_cases))


# ============================================================
# 4. ANSWER CORRECTNESS
# ============================================================

correctness = GEval(
    name="Answer Correctness",

    model="gpt-4.1",

    criteria="""
    Evaluate whether the generated answer is factually correct
    compared with the expected teacher answer.

    Give a high score when the important meaning and facts
    are correct.

    Penalize:
    - incorrect facts
    - contradictory information
    - missing important information

    Minor wording differences should not reduce the score.

    Focus on factual correctness and meaning rather than
    exact wording.
    """,

    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT
    ],

    async_mode=False
)


# ============================================================
# 5. ANSWER RELEVANCY
# ============================================================

relevancy = AnswerRelevancyMetric(
    model="gpt-4.1",

    include_reason=True,

    async_mode=False
)


# ============================================================
# 6. HELPFULNESS
# ============================================================

helpfulness = GEval(
    name="Helpfulness",

    model="gpt-4.1",

    criteria="""
    Evaluate how helpful the generated answer is for a
    beginner student.

    A helpful answer should:

    1. Directly answer the student's question.
    2. Be easy to understand.
    3. Provide enough explanation.
    4. Avoid unnecessary information.
    5. Help the student understand the concept.

    Give a high score when the answer is:
    - clear
    - beginner-friendly
    - relevant
    - educational
    - concise
    """,

    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT
    ],

    async_mode=False
)


# ============================================================
# 7. RUN DEEPEVAL
# ============================================================

print("\n" + "=" * 70)
print("STARTING DEEPEVAL")
print("=" * 70)

print("\nEvaluating:")
print("1. Answer Correctness")
print("2. Answer Relevancy")
print("3. Helpfulness")

print("\nPlease wait...\n")


result = evaluate(
    test_cases=test_cases,

    metrics=[
        correctness,
        relevancy,
        helpfulness
    ],

    async_config=AsyncConfig(
        run_async=False
    )
)


# ============================================================
# 8. COMPLETED
# ============================================================

print("\n" + "=" * 70)
print("DEEPEVAL COMPLETED")
print("=" * 70)

print("Result type:")
print(type(result))

print("\nAvailable result attributes:")
print(dir(result))

DATASET LOADED
Number of questions: 20
Columns: ['User Query', 'Teacher Response', 'GeneratedAnswer']

All required columns found.
Test cases created: 20

STARTING DEEPEVAL

Evaluating:
1. Answer Correctness
2. Answer Relevancy
3. Helpfulness

Please wait...



✨ You're running DeepEval's latest Answer Correctness [GEval] Metric! (using gpt-4.1, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1, strict=False, async_mode=False)...

✨ You're running DeepEval's latest Helpfulness [GEval] Metric! (using gpt-4.1, strict=False, async_mode=False)...

Output()

RetryError: RetryError[<Future at 0x28f5a953340 state=finished raised RateLimitError>]

In [10]:
result.test_results

NameError: name 'result' is not defined

In [13]:

for i, test_result in enumerate(result.test_results):

    metrics_data = test_result.metrics_data

    df.loc[i, "AnswerCorrectness"] = metrics_data[0].score
    df.loc[i, "AnswerRelevancy"] = metrics_data[1].score
    df.loc[i, "Helpfulness"] = metrics_data[2].score


# ============================================================
# 3. CALCULATE OVERALL ANSWER QUALITY
# ============================================================

df["AnswerQuality"] = (
    df["AnswerCorrectness"]
    + df["AnswerRelevancy"]
    + df["Helpfulness"]
) / 3


# ============================================================
# 4. ROUND SCORES
# ============================================================

metric_columns = [
    "AnswerCorrectness",
    "AnswerRelevancy",
    "Helpfulness",
    "AnswerQuality"
]

df[metric_columns] = df[metric_columns].round(3)


# ============================================================
# 5. AVERAGES
# ============================================================

average_correctness = df["AnswerCorrectness"].mean()
average_relevancy = df["AnswerRelevancy"].mean()
average_helpfulness = df["Helpfulness"].mean()
average_quality = df["AnswerQuality"].mean()


# ============================================================
# 6. QUESTION-WISE RESULTS
# ============================================================

print("\n")
print("=" * 120)
print("                         AI TEACHER ANSWER QUALITY")
print("=" * 120)

display_df = df[
    [
        "User Query",
        "AnswerCorrectness",
        "AnswerRelevancy",
        "Helpfulness",
        "AnswerQuality"
    ]
]


# ============================================================
# 8. SAVE FINAL RESULT
# ============================================================

df.to_csv(
    "ai_teacher_evaluation.csv",
    index=False,
    encoding="utf-8-sig"
)



NameError: name 'result' is not defined

In [14]:
# ============================================================
# DEEPEVAL - RUN EVALUATION AND SAVE SCORES
# ============================================================

import os
import pandas as pd

from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric, GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.evaluate import AsyncConfig


# ============================================================
# 1. CHECK OPENAI API KEY
# ============================================================

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY is not configured."
    )

print("OpenAI API key detected.")


# ============================================================
# 2. LOAD CSV
# ============================================================

df = pd.read_csv("ai_teacher_results.csv")

print("Dataset loaded.")
print("Questions:", len(df))


# ============================================================
# 3. CHECK COLUMNS
# ============================================================

required_columns = [
    "User Query",
    "Teacher Response",
    "GeneratedAnswer"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing columns: {missing}"
    )


# ============================================================
# 4. CREATE TEST CASES
# ============================================================

test_cases = []

for _, row in df.iterrows():

    test_cases.append(
        LLMTestCase(
            input=str(row["User Query"]),
            actual_output=str(row["GeneratedAnswer"]),
            expected_output=str(row["Teacher Response"])
        )
    )

print("Test cases created:", len(test_cases))


# ============================================================
# 5. CORRECTNESS METRIC
# ============================================================

correctness = GEval(
    name="Answer Correctness",

    model="gpt-4.1",

    criteria="""
    Evaluate whether the generated answer is factually correct
    compared with the expected teacher answer.

    Give a high score when the important meaning and facts
    are correct.

    Penalize incorrect facts, contradictory information,
    and missing important information.

    Minor wording differences should not reduce the score.
    """,

    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT
    ],

    async_mode=False
)


# ============================================================
# 6. RELEVANCY METRIC
# ============================================================

relevancy = AnswerRelevancyMetric(
    model="gpt-4.1",
    include_reason=True,
    async_mode=False
)


# ============================================================
# 7. HELPFULNESS METRIC
# ============================================================

helpfulness = GEval(
    name="Helpfulness",

    model="gpt-4.1",

    criteria="""
    Evaluate how helpful the generated answer is for a
    beginner student.

    A helpful answer should:

    1. Directly answer the student's question.
    2. Be easy to understand.
    3. Provide enough explanation.
    4. Avoid unnecessary information.
    5. Help the student understand the concept.

    Give a high score when the answer is clear,
    beginner-friendly, relevant, educational,
    and concise.
    """,

    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT
    ],

    async_mode=False
)


# ============================================================
# 8. RUN EVALUATION
# ============================================================

print("\n" + "=" * 70)
print("STARTING DEEPEVAL")
print("=" * 70)

print("Evaluating", len(test_cases), "questions...")
print("Please wait...\n")


# THIS LINE MUST RUN SUCCESSFULLY
result = evaluate(
    test_cases=test_cases,
    metrics=[
        correctness,
        relevancy,
        helpfulness
    ],
    async_config=AsyncConfig(
        run_async=False
    )
)


# ============================================================
# 9. RESULT CREATED
# ============================================================

print("\nEvaluation completed successfully.")

print("Result object:")
print(type(result))


# ============================================================
# 10. EXTRACT SCORES
# ============================================================

df["AnswerCorrectness"] = None
df["AnswerRelevancy"] = None
df["Helpfulness"] = None


for i, test_result in enumerate(result.test_results):

    for metric in test_result.metrics_data:

        metric_name = metric.name
        metric_score = metric.score

        print(
            f"Question {i + 1}: "
            f"{metric_name} = {metric_score}"
        )

        if metric_name == "Answer Correctness":
            df.loc[i, "AnswerCorrectness"] = metric_score

        elif metric_name == "Answer Relevancy":
            df.loc[i, "AnswerRelevancy"] = metric_score

        elif metric_name == "Helpfulness":
            df.loc[i, "Helpfulness"] = metric_score


# ============================================================
# 11. CONVERT SCORES TO NUMERIC
# ============================================================

score_columns = [
    "AnswerCorrectness",
    "AnswerRelevancy",
    "Helpfulness"
]

for column in score_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


# ============================================================
# 12. OVERALL SCORE
# ============================================================

df["OverallScore"] = df[
    score_columns
].mean(axis=1)


# ============================================================
# 13. SAVE FINAL CSV
# ============================================================

OUTPUT_FILE = "ai_teacher_deepeval_results.csv"

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 14. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)

display(
    df[
        [
            "User Query",
            "Teacher Response",
            "GeneratedAnswer",
            "AnswerCorrectness",
            "AnswerRelevancy",
            "Helpfulness",
            "OverallScore"
        ]
    ]
)


# ============================================================
# 15. AVERAGE SCORES
# ============================================================

print("\n" + "=" * 70)
print("AVERAGE SCORES")
print("=" * 70)

print(
    "Answer Correctness:",
    round(df["AnswerCorrectness"].mean(), 3)
)

print(
    "Answer Relevancy:",
    round(df["AnswerRelevancy"].mean(), 3)
)

print(
    "Helpfulness:",
    round(df["Helpfulness"].mean(), 3)
)

print(
    "Overall Score:",
    round(df["OverallScore"].mean(), 3)
)


print("\nFinal file saved as:")
print(OUTPUT_FILE)

OpenAI API key detected.
Dataset loaded.
Questions: 20
Test cases created: 20

STARTING DEEPEVAL
Evaluating 20 questions...
Please wait...



✨ You're running DeepEval's latest Answer Correctness [GEval] Metric! (using gpt-4.1, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1, strict=False, async_mode=False)...

✨ You're running DeepEval's latest Helpfulness [GEval] Metric! (using gpt-4.1, strict=False, async_mode=False)...

Output()

RetryError: RetryError[<Future at 0x2743cf1db10 state=finished raised RateLimitError>]

In [15]:
# ============================================================
# STAGE 3 - DEEPEVAL EVALUATION
# ONE QUESTION AT A TIME
# ============================================================

import os
import time
import pandas as pd

from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric, GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.evaluate import AsyncConfig


# ============================================================
# 1. CHECK OPENAI API KEY
# ============================================================

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY is not configured."
    )

print("OpenAI API key detected.")


# ============================================================
# 2. LOAD DATA
# ============================================================

INPUT_FILE = "ai_teacher_results.csv"
OUTPUT_FILE = "ai_teacher_deepeval_results.csv"

df = pd.read_csv(INPUT_FILE)

print("=" * 70)
print("DATASET LOADED")
print("=" * 70)

print("Total questions:", len(df))


# ============================================================
# 3. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "User Query",
    "Teacher Response",
    "GeneratedAnswer"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )


# ============================================================
# 4. CREATE RESULT COLUMNS
# ============================================================

df["AnswerCorrectness"] = None
df["AnswerRelevancy"] = None
df["Helpfulness"] = None
df["OverallScore"] = None


# ============================================================
# 5. CREATE METRICS
# ============================================================

correctness = GEval(
    name="Answer Correctness",

    model="gpt-4.1",

    criteria="""
    Evaluate whether the generated answer is factually correct
    compared with the expected teacher answer.

    Give a high score when the important meaning and facts
    are correct.

    Penalize:
    - incorrect facts
    - contradictory information
    - missing important information

    Minor wording differences should not reduce the score.
    """,

    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT
    ],

    async_mode=False
)


relevancy = AnswerRelevancyMetric(
    model="gpt-4.1",

    include_reason=False,

    async_mode=False
)


helpfulness = GEval(
    name="Helpfulness",

    model="gpt-4.1",

    criteria="""
    Evaluate how helpful the generated answer is for a
    beginner student.

    A helpful answer should:

    1. Directly answer the student's question.
    2. Be easy to understand.
    3. Provide enough explanation.
    4. Avoid unnecessary information.
    5. Help the student understand the concept.

    Give a high score when the answer is:
    - clear
    - beginner-friendly
    - relevant
    - educational
    - concise
    """,

    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT
    ],

    async_mode=False
)


metrics = [
    correctness,
    relevancy,
    helpfulness
]


# ============================================================
# 6. EVALUATE ONE QUESTION AT A TIME
# ============================================================

print("\n" + "=" * 70)
print("STARTING DEEPEVAL")
print("=" * 70)


for i, row in df.iterrows():

    print("\n" + "-" * 70)
    print(f"Processing question {i + 1}/{len(df)}")
    print(f"Question: {row['User Query']}")


    # --------------------------------------------------------
    # CREATE SINGLE TEST CASE
    # --------------------------------------------------------

    test_case = LLMTestCase(
        input=str(row["User Query"]),
        actual_output=str(row["GeneratedAnswer"]),
        expected_output=str(row["Teacher Response"])
    )


    # --------------------------------------------------------
    # RETRY CONTROL
    # --------------------------------------------------------

    max_retries = 3
    evaluation_success = False


    for attempt in range(max_retries):

        try:

            print(
                f"Evaluation attempt "
                f"{attempt + 1}/{max_retries}"
            )


            # ------------------------------------------------
            # RUN DEEPEVAL
            # ------------------------------------------------

            result = evaluate(
                test_cases=[test_case],

                metrics=metrics,

                async_config=AsyncConfig(
                    run_async=False
                )
            )


            # ------------------------------------------------
            # EXTRACT RESULT
            # ------------------------------------------------

            test_result = result.test_results[0]

            for metric in test_result.metrics_data:

                metric_name = metric.name
                metric_score = metric.score


                if metric_name == "Answer Correctness":

                    df.loc[
                        i,
                        "AnswerCorrectness"
                    ] = metric_score


                elif metric_name == "Answer Relevancy":

                    df.loc[
                        i,
                        "AnswerRelevancy"
                    ] = metric_score


                elif metric_name == "Helpfulness":

                    df.loc[
                        i,
                        "Helpfulness"
                    ] = metric_score


            # ------------------------------------------------
            # CALCULATE OVERALL SCORE
            # ------------------------------------------------

            scores = [
                df.loc[i, "AnswerCorrectness"],
                df.loc[i, "AnswerRelevancy"],
                df.loc[i, "Helpfulness"]
            ]

            valid_scores = [
                float(score)
                for score in scores
                if pd.notna(score)
            ]

            if valid_scores:

                df.loc[
                    i,
                    "OverallScore"
                ] = sum(valid_scores) / len(valid_scores)


            evaluation_success = True


            print("\nScores:")

            print(
                "Answer Correctness:",
                df.loc[i, "AnswerCorrectness"]
            )

            print(
                "Answer Relevancy:",
                df.loc[i, "AnswerRelevancy"]
            )

            print(
                "Helpfulness:",
                df.loc[i, "Helpfulness"]
            )

            print(
                "Overall Score:",
                df.loc[i, "OverallScore"]
            )


            break


        except Exception as e:

            print(
                f"\nAttempt {attempt + 1} failed:"
            )

            print(str(e))


            if attempt < max_retries - 1:

                wait_time = 10 * (attempt + 1)

                print(
                    f"Waiting {wait_time} seconds "
                    "before retry..."
                )

                time.sleep(wait_time)


            else:

                print(
                    "Maximum retries reached."
                )

                print(
                    "This question will be skipped."
                )


    # --------------------------------------------------------
    # SAVE AFTER EVERY QUESTION
    # --------------------------------------------------------

    df.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"\nProgress saved: "
        f"{i + 1}/{len(df)}"
    )


    # --------------------------------------------------------
    # WAIT BEFORE NEXT QUESTION
    # --------------------------------------------------------

    if evaluation_success:

        print(
            "Waiting before next question..."
        )

        time.sleep(5)


# ============================================================
# 7. FINAL SAVE
# ============================================================

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 8. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("DEEPEVAL COMPLETED")
print("=" * 70)


print(
    "Average Answer Correctness:",
    round(
        pd.to_numeric(
            df["AnswerCorrectness"],
            errors="coerce"
        ).mean(),
        3
    )
)


print(
    "Average Answer Relevancy:",
    round(
        pd.to_numeric(
            df["AnswerRelevancy"],
            errors="coerce"
        ).mean(),
        3
    )
)


print(
    "Average Helpfulness:",
    round(
        pd.to_numeric(
            df["Helpfulness"],
            errors="coerce"
        ).mean(),
        3
    )
)


print(
    "Overall Average Score:",
    round(
        pd.to_numeric(
            df["OverallScore"],
            errors="coerce"
        ).mean(),
        3
    )
)


print("\nFinal file:")
print(OUTPUT_FILE)


# ============================================================
# 9. DISPLAY RESULTS
# ============================================================

display(
    df[
        [
            "User Query",
            "Teacher Response",
            "GeneratedAnswer",
            "AnswerCorrectness",
            "AnswerRelevancy",
            "Helpfulness",
            "OverallScore"
        ]
    ]
)

OpenAI API key detected.
DATASET LOADED
Total questions: 20

STARTING DEEPEVAL

----------------------------------------------------------------------
Processing question 1/20
Question: What is AI?
Evaluation attempt 1/3


✨ You're running DeepEval's latest Answer Correctness [GEval] Metric! (using gpt-4.1, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1, strict=False, async_mode=False)...

✨ You're running DeepEval's latest Helpfulness [GEval] Metric! (using gpt-4.1, strict=False, async_mode=False)...

Output()

...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1, strict=False, async_mode=False)...

✨ You're running DeepEval's latest Helpfulness [GEval] Metric! (using gpt-4.1, strict=False, async_mode=False)...

Output()


Attempt 3 failed:
RetryError[<Future at 0x2743d09fe80 state=finished raised RateLimitError>]
Maximum retries reached.
This question will be skipped.

Progress saved: 20/20

DEEPEVAL COMPLETED
Average Answer Correctness: nan
Average Answer Relevancy: nan
Average Helpfulness: nan
Overall Average Score: nan

Final file:
ai_teacher_deepeval_results.csv


,User Query,Teacher Response,GeneratedAnswer,AnswerCorrectness,AnswerRelevancy,Helpfulness,OverallScore
0,What is AI?,AI (Artificial Intelligence) is the ability of...,ERROR: 503 UNAVAILABLE. {'error': {'code': 503...,None,None,None,None
1,How does AI differ from traditional programming?,Traditional programs follow fixed rules writte...,Input + Rules = Output).,None,None,None,None
2,What is Machine Learning?,Machine Learning is a branch of AI where compu...,ERROR: 503 UNAVAILABLE. {'error': {'code': 503...,None,None,None,None
3,What is Deep Learning?,Deep Learning is a type of Machine Learning th...,Deep learning is a,None,None,None,None
4,"What is the difference between AI, ML, and Dee...",AI is the broad field of making machines intel...,ERROR: 503 UNAVAILABLE. {'error': {'code': 503...,None,None,None,None
5,What is training data?,Training data is the data used to teach a Mach...,ERROR: 503 UNAVAILABLE. {'error': {'code': 503...,None,None,None,None
6,What is a Machine Learning model?,A Machine Learning model is a mathematical sys...,ERROR: 503 UNAVAILABLE. {'error': {'code': 503...,None,None,None,None
7,What is supervised learning?,Supervised learning trains a model using examp...,It is a,None,None,None,None
8,What is unsupervised learning?,Unsupervised learning works with data that doe...,Unsupervised learning is when,None,None,None,None
9,What is reinforcement learning?,Reinforcement learning teaches an agent throug...,ERROR: 503 UNAVAILABLE. {'error': {'code': 503...,None,None,None,None


In [8]:
df

,User Query,Teacher Response,GeneratedAnswer,AnswerCorrectness,AnswerRelevancy,Helpfulness,AnswerQuality
0,What is AI?,AI (Artificial Intelligence) is the ability of...,AI (Artificial Intelligence) is technology tha...,0.890,1.000,1.000,0.963
1,How does AI differ from traditional programming?,Traditional programs follow fixed rules writte...,Traditional programming follows explicit rules...,1.000,1.000,1.000,1.000
2,What is Machine Learning?,Machine Learning is a branch of AI where compu...,Machine Learning is a way for computers to lea...,0.900,1.000,1.000,0.967
3,What is Deep Learning?,Deep Learning is a type of Machine Learning th...,Deep Learning is a type of AI that uses artifi...,0.729,1.000,1.000,0.910
4,"What is the difference between AI, ML, and Dee...",AI is the broad field of making machines intel...,AI is the broad idea of making computers perfo...,0.900,1.000,1.000,0.967
5,What is training data?,Training data is the data used to teach a Mach...,Training data is the information used to teach...,0.900,1.000,1.000,0.967
6,What is a Machine Learning model?,A Machine Learning model is a mathematical sys...,A machine learning model is a computer program...,0.900,1.000,1.000,0.967
7,What is supervised learning?,Supervised learning trains a model using examp...,Supervised learning is a type of machine learn...,0.900,1.000,0.998,0.966
8,What is unsupervised learning?,Unsupervised learning works with data that doe...,Unsupervised learning is a type of machine lea...,0.969,1.000,1.000,0.990
9,What is reinforcement learning?,Reinforcement learning teaches an agent throug...,Reinforcement learning is a type of machine le...,1.000,1.000,0.900,0.967


In [15]:
average_result=pd.DataFrame(columns=["average_correctness","average_relevancy","average_helpfulness","average_quality"],
                            data=[[average_correctness,average_relevancy,average_helpfulness,average_quality]])

In [16]:
average_result

,average_correctness,average_relevancy,average_helpfulness,average_quality
0,0.87875,1.0,0.9985,0.9591
